# H5 dense 접두 예상 효과의 상한: 기준이 놓친 정답의 메타데이터 일치

dense 접두(청크 앞에 날짜·위원회·회의명·회차·안건을 붙여 재임베딩)는 질의가 메타데이터로 회의를 지목하고, 그 메타데이터가 정답을 경쟁 문서와 가를 수 있는 실패만 고칠 수 있다. 재임베딩 전에 EXP-014(D-32)가 놓친 정답을 질의 조건(H14 추출기)과 정답 문서의 일치로 나눠 그런 실패가 얼마나 남았는지 센다. 정답 라벨은 실패를 찾는 데만 쓴다.

In [1]:
import json

import pandas as pd

from rag.conditions import ConditionIndex, build_extractor
from rag.eval.metrics import load_gold, read_run
from rag.eval_multi.metrics import load_gold as load_multi
from rag.eval_multi.metrics import ranked_docs
from rag.index import ROOT, load_cfg
from rag.search import query_list, read_texts

C:\workspace\meet-rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RUN = 'data/runs/snowflake-ko-fixed-512-64-hybrid-cond-filter-law-ms256-k20'
TOP = 5

In [3]:
cfg = load_cfg(['+exp=exp014', 'index.scope=full'])
c, rc = cfg.conditions, cfg.retriever.conditions
with (ROOT / cfg.paths.corpus).open(encoding='utf-8') as f:
    docs = [json.loads(x) for x in f]
pos = {d['doc_id']: i for i, d in enumerate(docs)}
cidx = ConditionIndex(docs, rc.year_ref_offsets,
                      c.law_min_chars if c.get('agenda_law') else None)
ex = build_extractor(cfg)
print(len(docs), 'docs, min_fields', rc.min_fields)

38516 docs, min_fields 2


## 분류

- 조건 없음: 질의에서 메타데이터 조건이 뽑히지 않음
- 2필드 미만: 조건은 있지만 어떤 문서도 `min_fields`개 이상 맞지 않음
- 최다 일치(동률 수): 정답이 맞은 필드 수가 가장 많은 문서이고, 같은 수만큼 맞은 문서가 함께 있음
- 일치 적음: 정답보다 더 많이 맞은 문서가 있음
- 정답 불일치: 정답이 조건에 하나도 맞지 않음

In [4]:
def kind(cond, gold_doc):
    if not cond:
        return '조건 없음'
    counts = cidx.match_counts(cond)
    if not counts or max(counts.values()) < rc.min_fields:
        return f'{rc.min_fields}필드 미만'
    g, top = counts.get(pos[gold_doc], 0), max(counts.values())
    n_best = sum(v == top for v in counts.values())
    if g == top:
        return f"최다 일치(동률 {'>5' if n_best > 5 else '≤5'})"
    return '일치 적음' if g > 0 else '정답 불일치'

## single-doc: dev-full 상위 5개 실패

In [5]:
gold = load_gold(cfg, 'dev-full')
run = read_run(ROOT / f'{RUN}-dev-full.csv').sort_values(['qid', 'rank'])
top = run.groupby('qid')['doc_id'].apply(list)
qids, path = query_list(cfg)
texts = dict(zip(qids, read_texts(path, qids), strict=True))
miss = [q for q in qids if gold.loc[q, 'doc_id'] not in top[q][:TOP]]
single = pd.Series([kind(ex.extract(texts[q]), gold.loc[q, 'doc_id']) for q in miss])
print('실패', len(miss), '/', len(qids))
print(single.value_counts().to_string())

실패 177 / 3016
조건 없음     166
2필드 미만     10
일치 적음       1


## multi-doc: 상위 5개 밖 정답 문서

In [6]:
mcfg = load_cfg(['+exp=exp014', 'index.scope=full', 'search.queries=multidoc'])
mgold = load_multi(ROOT / mcfg.paths.multidoc_frozen)
mq, mpath = query_list(mcfg)
mtexts = dict(zip(mq, read_texts(mpath, mq), strict=True))
mtop = ranked_docs(read_run(ROOT / f'{RUN}-multidoc.csv'), mgold, 10)
rows = []
for q in mq:
    cond = ex.extract(mtexts[q])
    for g in mgold.loc[q, 'gold']:
        if g not in mtop[q][:TOP]:
            rows.append({'type': mgold.loc[q, 'type'], 'kind': kind(cond, g),
                         'in10': g in mtop[q][:10]})
md_miss = pd.DataFrame(rows)
print('정답 문서', sum(len(s) for s in mgold['gold']), ', 상위 5 밖', len(md_miss),
      ', 그중 10위 안', int(md_miss['in10'].sum()))
print(pd.crosstab(md_miss['kind'], md_miss['type'], margins=True).to_string())

정답 문서 1491 , 상위 5 밖 84 , 그중 10위 안 26
type          conf  law  questioner  All
kind                                    
일치 적음            5   25           2   32
최다 일치(동률 >5)    11   13          28   52
All             16   38          30   84
